In [6]:
import pandas as pd
import glob
import os
import warnings

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

In [7]:
# =========================================================================
# ⚙️ DATA QUALITY FUNCTION: PRE-MERGE VALIDATION
# =========================================================================

DATE_FIELDS    = ['CloseDate', 'ListingContractDate',
                  'PurchaseContractDate', 'ContractStatusChangeDate']
NUMERIC_FIELDS = ['ClosePrice', 'ListPrice', 'OriginalListPrice',
                  'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger']
POSTAL_FIELD   = 'PostalCode'
ROW_COUNT_FLOOR = 100


def check_file_quality(file_paths, label):
    """
    Pre-merge validation summarised by check category, not by file.
    Prints one block per check; expands details only when issues exist.
    All files are always returned — nothing is silently dropped.
    """
    if not file_paths:
        print(f"❌ No {label} files found — aborting.")
        return []

    n = len(file_paths)
    print("=" * 60)
    print(f"PRE-MERGE QUALITY CHECK: {label.upper()} FILES ({n} files)")
    print("=" * 60)

    # Baseline columns from the first file
    baseline_cols = set(
        pd.read_csv(file_paths[0], nrows=5, low_memory=False).columns
    )

    # ── Collect results for all three checks ──
    col_issues  = {}   # {filename: {missing, extra}}
    row_issues  = {}   # {filename: [issue strings]}
    fmt_issues  = {}   # {filename: [issue strings]}

    for f_path in file_paths:
        f_name = os.path.basename(f_path)
        try:
            df_head = pd.read_csv(f_path, nrows=5, low_memory=False)
            df_full = pd.read_csv(
                f_path,
                dtype={POSTAL_FIELD: str, 'ListingKey': str},
                low_memory=False
            )

            # CHECK 1: Column Structure
            current_cols = set(df_head.columns)
            missing = baseline_cols - current_cols
            extra   = current_cols  - baseline_cols
            if missing or extra:
                col_issues[f_name] = {'missing': sorted(missing),
                                      'extra':   sorted(extra)}

            # CHECK 2: Row Structure
            r_issues = []
            total_rows = len(df_full)
            dup_rows   = df_full.duplicated().sum()
            if total_rows == 0:
                r_issues.append("Empty file (0 rows)")
            elif total_rows < ROW_COUNT_FLOOR:
                r_issues.append(f"Only {total_rows} rows (threshold: {ROW_COUNT_FLOOR})")
            if dup_rows > 0:
                r_issues.append(f"{dup_rows:,} fully duplicate rows")
            if r_issues:
                row_issues[f_name] = r_issues

            # CHECK 3: Data Format
            f_issues = []
            for col in DATE_FIELDS:
                if col in df_full.columns:
                    sample = df_full[col].dropna().head(50)
                    if len(sample) > 0:
                        try:
                            pd.to_datetime(sample)
                        except Exception:
                            f_issues.append(f"'{col}' cannot be parsed as date")

            for col in NUMERIC_FIELDS:
                if col in df_full.columns:
                    non_num = (
                        pd.to_numeric(df_full[col], errors='coerce').isna().sum()
                        - df_full[col].isna().sum()
                    )
                    if non_num > 0:
                        f_issues.append(f"'{col}' has {non_num:,} non-numeric values")

            if POSTAL_FIELD in df_full.columns:
                bad = df_full[POSTAL_FIELD].dropna()
                bad = bad[~bad.str.match(r'^\d{5}$')]
                if len(bad) > 0:
                    f_issues.append(f"PostalCode: {len(bad):,} values not 5-digit")

            if f_issues:
                fmt_issues[f_name] = f_issues

        except Exception as e:
            row_issues[f_name] = [f"Read error: {str(e)}"]

    # ── Print CHECK 1 ──
    if col_issues:
        print(f"[1] Column Structure  ...  ⚠️  {len(col_issues)} file(s) have column differences")
        for fname, diff in col_issues.items():
            print(f"      • {fname}")
            if diff['missing']:
                print(f"          Missing : {diff['missing']}")
            if diff['extra']:
                print(f"          Extra   : {diff['extra']}")
    else:
        print(f"[1] Column Structure  ...  ✅  All {n} files passed")

    # ── Print CHECK 2 ──
    if row_issues:
        print(f"[2] Row Structure     ...  ⚠️  {len(row_issues)} file(s) flagged")
        for fname, issues in row_issues.items():
            for iss in issues:
                print(f"      • {fname}   {iss}")
    else:
        print(f"[2] Row Structure     ...  ✅  All {n} files passed")

    # ── Print CHECK 3 ──
    if fmt_issues:
        print(f"[3] Data Format       ...  ⚠️  {len(fmt_issues)} file(s) flagged")
        for fname, issues in fmt_issues.items():
            for iss in issues:
                print(f"      • {fname}   {iss}")
    else:
        print(f"[3] Data Format       ...  ✅  All {n} files passed")

    # ── Summary line ──
    all_flagged = set(col_issues) | set(row_issues) | set(fmt_issues)
    n_clean     = n - len(all_flagged)
    print(f"\nRESULT: {n_clean} clean / {len(all_flagged)} flagged "
          f"— all {n} files included in merge")
    print("=" * 60 + "\n")

    return file_paths


# =========================================================================
# 🚀 MAIN DATA PIPELINE
# =========================================================================

PATH = r'C:\Users\lidon\Desktop\2026spring\IDX_MLS_Analytics'

sold_files    = sorted(glob.glob(os.path.join(PATH, 'CRMLSSold*.csv')))
listing_files = sorted(glob.glob(os.path.join(PATH, 'CRMLSListing*.csv')))

# -------------------------------------------------------------------------
# 1. Sold Data Quality Validation & Processing
# -------------------------------------------------------------------------
safe_sold_files = check_file_quality(sold_files, label='Sold')

sold_list = []
for f in safe_sold_files:
    df = pd.read_csv(
        f,
        dtype={POSTAL_FIELD: str, 'ListingKey': str},
        low_memory=False
    )
    sold_list.append(df)

if sold_list:
    combined_sold      = pd.concat(sold_list, ignore_index=True, join='outer')
    sold_before_filter = len(combined_sold)
    combined_sold      = combined_sold[combined_sold['PropertyType'] == 'Residential']
    sold_after_filter  = len(combined_sold)
else:
    combined_sold = pd.DataFrame()
    sold_before_filter = sold_after_filter = 0

# -------------------------------------------------------------------------
# 2. Listing Data Quality Validation & Processing
# -------------------------------------------------------------------------
safe_listing_files = check_file_quality(listing_files, label='Listing')

listing_list = []
for f in safe_listing_files:
    df = pd.read_csv(
        f,
        dtype={POSTAL_FIELD: str, 'ListingKey': str},
        low_memory=False
    )
    listing_list.append(df)

if listing_list:
    combined_listings      = pd.concat(listing_list, ignore_index=True, join='outer')
    listings_before_filter = len(combined_listings)
    combined_listings      = combined_listings[combined_listings['PropertyType'] == 'Residential']
    listings_after_filter  = len(combined_listings)
else:
    combined_listings = pd.DataFrame()
    listings_before_filter = listings_after_filter = 0

# -------------------------------------------------------------------------
# 3. Output
# -------------------------------------------------------------------------
combined_sold.to_csv(
    os.path.join(PATH, 'all_sold_residential.csv'), index=False, encoding='utf-8'
)
combined_listings.to_csv(
    os.path.join(PATH, 'all_listings_residential.csv'), index=False, encoding='utf-8'
)

# -------------------------------------------------------------------------
# 4. Final Verification Report
# -------------------------------------------------------------------------
print("=================== IDX PIPELINE EXECUTION LOG ===================")
print(f"  Sold    files processed    : {len(safe_sold_files)}")
print(f"  Listing files processed    : {len(safe_listing_files)}")
print(f"  Sold    rows before filter : {sold_before_filter:,}")
print(f"  Sold    rows after  filter : {sold_after_filter:,}")
print(f"  Sold    rows removed       : {sold_before_filter - sold_after_filter:,}")
print(f"  Listing rows before filter : {listings_before_filter:,}")
print(f"  Listing rows after  filter : {listings_after_filter:,}")
print(f"  Listing rows removed       : {listings_before_filter - listings_after_filter:,}")
print("==================================================================")

PRE-MERGE QUALITY CHECK: SOLD FILES (29 files)
[1] Column Structure  ...  ⚠️  26 file(s) have column differences
      • CRMLSSold202402.csv
          Missing : ['latfilled', 'lonfilled']
      • CRMLSSold202405.csv
          Missing : ['BuyerAgencyCompensation', 'BuyerAgencyCompensationType']
          Extra   : ['BuyerAgentAOR', 'ListAgentAOR']
      • CRMLSSold202406.csv
          Missing : ['BuyerAgencyCompensation', 'BuyerAgencyCompensationType']
          Extra   : ['BuyerAgentAOR', 'ListAgentAOR']
      • CRMLSSold202407.csv
          Missing : ['BuyerAgencyCompensation', 'BuyerAgencyCompensationType']
          Extra   : ['BuyerAgentAOR', 'ListAgentAOR']
      • CRMLSSold202408.csv
          Missing : ['BuyerAgencyCompensation', 'BuyerAgencyCompensationType', 'latfilled', 'lonfilled']
          Extra   : ['BuyerAgentAOR', 'ListAgentAOR']
      • CRMLSSold202409.csv
          Missing : ['BuyerAgencyCompensation', 'BuyerAgencyCompensationType', 'latfilled', 'lonfilled']
         